This is part of the materials for Exercise Session 1 of the Reinforcement Learning course (EL2805) at KTH, Stockholm. For questions and feedback, please contact one of the course TAs.

# 🎓 **<font color='teal'>Hands-On: When Supervised Learning performs well</font>**

This script demonstrates how to perform imitation learning using the Stable Baselines library and Hugging Face's model hub. It includes steps for setting up the environment, loading an expert policy, collecting demonstrations, and training an agent.

The script is already fully completed. Your task is to familiarize yourself with the tools used and to be able to reuse the code for similar tasks. The code provided is based on a tutorial from the Imitation library documentation (link below).


### 🛠️ **<font color='teal'>Tools Used:</font>**
- **Imitation Library**: A library for imitation learning algorithms.
- **Stable Baselines3**: A set of reliable implementations of reinforcement learning algorithms.
- **Hugging Face**: A hub for sharing and downloading pre-trained models.


### 🌄 **<font color='teal'>Environment:</font>** MountainCar
 - [Environment description](https://huggingface.co/HumanCompatibleAI/ppo-seals-MountainCar-v0)


### 📚 **References:**
- [Imitation Library Documentation](https://imitation.readthedocs.io/en/latest/tutorials/1_train_bc.html)
- [Stable Baselines3 Documentation](https://stable-baselines3.readthedocs.io/en/master/)
- [Hugging Face Model Hub](https://huggingface.co/)
  
To speed up training, you should use a GPU. In Colab, go to Runtime → Change runtime type → Hardware accelerator and select a GPU. Once enabled, PyTorch will automatically detect the GPU with torch.cuda.is_available(). Free Colab accounts have a limited number of GPU hours per week, so plan your experiments accordingly. All experiments in this course can also be run on a CPU, though training may be slower.


## **<font color='gray'>Step 1</font>**: <font color='teal'>Install Required Libraries</font>


In [ ]:
# Install imitation library containing behavior cloning algorithm
!pip install imitation

# Install system dependencies for environment rendering
# This is usually needed when working with notebooks (in Colab or in Jupyter)
# Not needed when rendering in Python IDE
!apt-get install python-opengl -y
!apt install xvfb -y
!pip install pyvirtualdisplay
!pip install piglet


## **<font color='gray'>Step 2</font>**: <font color='teal'>Load Libraries and Functions</font>


In [2]:
# General libraries
import numpy as np
import torch

# RL libraries
import gymnasium as gym
from imitation.policies.serialize import load_policy
from imitation.util.util import make_vec_env
from imitation.data.wrappers import RolloutInfoWrapper
from stable_baselines3.common.evaluation import evaluate_policy

# Rendering libraries
from pyvirtualdisplay import Display
from IPython import display
import time
import matplotlib.pyplot as plt
%matplotlib inline


## **<font color='gray'>Step 3</font>**: <font color='teal'>Set Up the Environment</font>


In [3]:
# Creating a vectorized environment for MountainCar-v0
# Vectorized environments allow for running multiple instances of the same environment in parallel
# This can speed up training of RL algos

env_MountainCar = make_vec_env(
    "seals:seals/MountainCar-v0", # seals is a library that contains some of the RL environments
    rng=np.random.default_rng(),
    post_wrappers=[
        lambda env_MountainCar, _: RolloutInfoWrapper(env_MountainCar)
    ],  # needed for computing rollouts later, can read more about wrappers here: https://gymnasium.farama.org/api/wrappers/
)


## **<font color='gray'>Step 4</font>**: <font color='teal'>Load Expert Policy</font>


In [ ]:
# Load an expert policy from Hugging Face (platform with trained models, data sets etc)
# We use a PPO policy (we will cover this algo later in the course) trained on MountainCar-v0
# Here you could also load you own policies and test them! Feel free to try that out - a lot of tutorials available online!

expert_MountainCar = load_policy(
    "ppo-huggingface",
    organization="HumanCompatibleAI",
    env_name="seals/MountainCar-v0",
    venv=env_MountainCar,
)

## **<font color='gray'>Step 5</font>**: <font color='teal'>Rendering Function</font>


In [5]:
# We will call this function later for rendering the environment
# It is fairly simple - the difficult part is rendering in Colab

def render_environment(policy, env_name='MountainCar-v0', num_steps=200):
    """
    Renders the specified environment using the provided policy.

    Parameters:
    - policy: The trained policy to use for selecting actions.
    - env_name: The name of the gym environment (default is 'MountainCar-v0').
    - num_steps: The number of steps to run the simulation.
    """
    # Start the virtual display
    display_window = Display().start()

    # Create the environment
    env = gym.make(env_name, render_mode='rgb_array')

    # Reset the environment to get the initial observation
    obs, _ = env.reset()

    # Set up the plot for rendering
    plt.axis('off')  # Turn off the axis
    img = plt.imshow(env.render())  # Initialize the image display

    done = False # if the goal has been reached this flag becomes True
    terminated = False # if the number of steps is larger than 200 this becomes True

    # Run for a fixed number of steps
    while(not(done) and not(terminated)):
        # Update the display with the latest frame
        img.set_data(env.render())  # Update the image data
        display.display(plt.gcf())  # Display the updated figure
        time.sleep(0.1)

        # Convert the observation to a tensor and add a batch dimension
        obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)

        # Get the action from the policy (make sure to use .detach() if necessary)
        act_tensor = policy(obs_tensor)[0]

        # Take a step in the environment with the selected action
        obs, _, done, terminated, _ = env.step(act_tensor.detach().numpy()[0])  # Convert to NumPy array

        display.clear_output(wait=True)  # Clear the output to avoid flickering

    # Close the environment after the episode
    env.close()
    display_window.stop()  # Properly close the display after rendering


## **<font color='gray'>Step 6</font>**: <font color='teal'>Evaluating Expert Policy</font>


In [ ]:
# Now that we have loaded the expert policy, we can test it!
expert_MountainCar.to('cpu')

# First let us take a look how it behaves
render_environment(expert_MountainCar)

# Next we can also quantify how good it is?
total_reward, _ = evaluate_policy(expert_MountainCar, env_MountainCar, 10)
print(f"Reward with expert policy: {total_reward}")

# Take a look at documentation files for MountainCar environment and make sure this is indeed good total reward

## **<font color='gray'>Step 7</font>**: <font color='teal'>Collect Data with Expert Policy</font>



As we mentioned in the exercise materials, we are not provided directly with expert policy $\pi^\star$, but instead only with data generated from it - states sampled according to $s\sim d^\star$ and actions $a\sim \pi^\star(s)$. In this step we are going to generate these data points from trajectories:

In [7]:
# Generate rollouts (demonstrations) using the expert policy

from imitation.data import rollout
rng = np.random.default_rng()

# We collect at least trajectories from 500 episodes:
rollouts = rollout.rollout(
    expert_MountainCar,
    env_MountainCar,
    rollout.make_sample_until(min_timesteps=None, min_episodes=500),
    rng=rng,
)

# Lastly, we flatten the collected rollouts into transitions (s1,a1,s1'), (s2,a2,s2'),...
transitions = rollout.flatten_trajectories(rollouts)
print(vars(transitions).keys())

## **<font color='gray'>Step 8</font>**: <font color='teal'>Train the Imitation Learning Agent</font>


The algorithm that we used in the exercise materials is called behavior cloning and here we train a policy $\widehat{\pi}$ using this algorithm:

In [ ]:
# Using behavior cloning (BC) to train the agent
from imitation.algorithms import bc

bc_trainer = bc.BC(
    observation_space=env_MountainCar.observation_space,
    action_space=env_MountainCar.action_space,
    demonstrations=transitions,
    rng=rng,
)

# Train the agent with behavior cloning
bc_trainer.train(n_epochs=1)

## **<font color='gray'>Step 9</font>**: <font color='teal'>Evaluate Imitating Agent</font>


Finally, we can evaluate performance of our imitating agent. What do you expect - can it achieve better policy than the expert policy we started with? Why/Why not? Compare renderings - is there any qualitative difference?

In [ ]:
bc_trainer.policy.to('cpu')

# Evaluate the agent after training
render_environment(bc_trainer.policy)

reward_after_training, _ = evaluate_policy(bc_trainer.policy, env_MountainCar, 10)
print(f"Reward after training: {reward_after_training}")